# 07. 대시보드용 데이터 생성

분석이 완료된 고객별 프로모션 수신 데이터를 Tableau에서 사용할 수 있도록 정리한다.

생성 파일:

1. `dashboard_detail.csv`: 고객별 프로모션 수신 건
2. `dashboard_offer_summary.csv`: Offer별 성과
3. `dashboard_segment_summary.csv`: 고객군·Offer 유형별 성과

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# 저장 폴더 생성
output_dir = Path("dashboard_data")
output_dir.mkdir(exist_ok=True)

# 분석 데이터 불러오기
df = pd.read_pickle("customer_analysis.pkl")

# -------------------------------------
# 1. 상세 데이터
# -------------------------------------

dashboard_detail = df.copy()

# Tableau에서 사용할 수 있도록 채널 목록을 문자열로 변환
dashboard_detail["channel_combination"] = (
    dashboard_detail["channels"]
    .apply(lambda values: ", ".join(values))
)

# 채널 포함 여부
for channel in ["web", "email", "mobile", "social"]:
    dashboard_detail[f"has_{channel}"] = (
        dashboard_detail["channels"]
        .apply(lambda values: int(channel in values))
    )

# 0/1 지표
dashboard_detail["viewed_int"] = (
    dashboard_detail["viewed"].astype(int)
)

dashboard_detail["completed_after_view"] = (
    dashboard_detail["viewed_before_completed"].astype(int)
)

dashboard_detail["completed_without_prior_view_int"] = (
    dashboard_detail["completed_without_prior_view"].astype(int)
)

# Tableau용 고객 구간 재생성
dashboard_detail["gender_dashboard"] = np.select(
    [
        dashboard_detail["gender"].eq("F"),
        dashboard_detail["gender"].eq("M"),
        dashboard_detail["gender"].eq("O")
    ],
    ["여성", "남성", "기타"],
    default="정보 없음"
)

dashboard_detail["age_dashboard"] = np.select(
    [
        dashboard_detail["age"].eq(118),
        dashboard_detail["age"].le(54),
        dashboard_detail["age"].ge(55)
    ],
    ["정보 없음", "54세 이하", "55세 이상"],
    default="정보 없음"
)

dashboard_detail["income_dashboard"] = np.select(
    [
        dashboard_detail["income"].isna(),
        dashboard_detail["income"].lt(80000),
        dashboard_detail["income"].ge(80000)
    ],
    ["정보 없음", "8만 달러 미만", "8만 달러 이상"],
    default="정보 없음"
)

# channels 원본 리스트 열은 CSV에서 제외
dashboard_detail = dashboard_detail.drop(columns=["channels"])

# -------------------------------------
# 2. Offer별 요약 데이터
# -------------------------------------

dashboard_offer_summary = (
    dashboard_detail.groupby(
        [
            "offer_id",
            "offer_type",
            "reward",
            "difficulty",
            "duration",
            "channel_combination"
        ],
        as_index=False
    )
    .agg(
        received_count=("offer_id", "size"),
        viewed_count=("viewed_int", "sum"),
        completed_after_view_count=("completed_after_view", "sum"),
        completed_without_prior_view_count=(
            "completed_without_prior_view_int",
            "sum"
        )
    )
)

dashboard_offer_summary["view_rate"] = (
    dashboard_offer_summary["viewed_count"]
    / dashboard_offer_summary["received_count"]
)

dashboard_offer_summary["completion_after_view_rate"] = (
    dashboard_offer_summary["completed_after_view_count"]
    / dashboard_offer_summary["viewed_count"]
)

# 정보 제공형은 완료 개념이 없으므로 N/A 처리
dashboard_offer_summary.loc[
    dashboard_offer_summary["offer_type"].eq("informational"),
    "completion_after_view_rate"
] = np.nan

# -------------------------------------
# 3. 고객군·Offer 유형별 요약
# -------------------------------------

segment_frames = []

segment_settings = [
    (
        "성별",
        "gender_dashboard",
        dashboard_detail["gender_dashboard"].isin(["여성", "남성"])
    ),
    (
        "연령",
        "age_dashboard",
        dashboard_detail["age_dashboard"].ne("정보 없음")
    ),
    (
        "소득",
        "income_dashboard",
        dashboard_detail["income_dashboard"].ne("정보 없음")
    )
]

for segment_type, segment_column, condition in segment_settings:
    segment_data = dashboard_detail.loc[condition].copy()

    segment_summary = (
        segment_data.groupby(
            [segment_column, "offer_type"],
            as_index=False
        )
        .agg(
            received_count=("offer_id", "size"),
            viewed_count=("viewed_int", "sum"),
            completed_after_view_count=(
                "completed_after_view",
                "sum"
            )
        )
        .rename(columns={segment_column: "segment_value"})
    )

    segment_summary["segment_type"] = segment_type

    segment_frames.append(segment_summary)

dashboard_segment_summary = pd.concat(
    segment_frames,
    ignore_index=True
)

dashboard_segment_summary["view_rate"] = (
    dashboard_segment_summary["viewed_count"]
    / dashboard_segment_summary["received_count"]
)

dashboard_segment_summary["completion_after_view_rate"] = (
    dashboard_segment_summary["completed_after_view_count"]
    / dashboard_segment_summary["viewed_count"]
)

dashboard_segment_summary.loc[
    dashboard_segment_summary["offer_type"].eq("informational"),
    "completion_after_view_rate"
] = np.nan

# 열 순서 정리
dashboard_segment_summary = dashboard_segment_summary[
    [
        "segment_type",
        "segment_value",
        "offer_type",
        "received_count",
        "viewed_count",
        "completed_after_view_count",
        "view_rate",
        "completion_after_view_rate"
    ]
]

# -------------------------------------
# 4. CSV 저장
# -------------------------------------

dashboard_detail.to_csv(
    output_dir / "dashboard_detail.csv",
    index=False,
    encoding="utf-8-sig"
)

dashboard_offer_summary.to_csv(
    output_dir / "dashboard_offer_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

dashboard_segment_summary.to_csv(
    output_dir / "dashboard_segment_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# -------------------------------------
# 5. 결과 검수
# -------------------------------------

assert len(dashboard_detail) == 66041
assert dashboard_detail["viewed_int"].sum() == 49417

assert len(dashboard_offer_summary) == 10
assert dashboard_offer_summary["received_count"].sum() == 66041
assert dashboard_offer_summary["viewed_count"].sum() == 49417

assert (
    dashboard_segment_summary.loc[
        dashboard_segment_summary["segment_type"].eq("성별"),
        "received_count"
    ].sum()
    == 56752
)

assert (
    dashboard_segment_summary.loc[
        dashboard_segment_summary["segment_type"].eq("연령"),
        "received_count"
    ].sum()
    == 57561
)

assert (
    dashboard_segment_summary.loc[
        dashboard_segment_summary["segment_type"].eq("소득"),
        "received_count"
    ].sum()
    == 57561
)

print("[저장 완료]")
print("상세 데이터:", dashboard_detail.shape)
print("Offer 요약:", dashboard_offer_summary.shape)
print("고객군 요약:", dashboard_segment_summary.shape)

print("\n[주요 지표 검수]")
print("전체 수신 건수:", len(dashboard_detail))
print("전체 열람 건수:", dashboard_detail["viewed_int"].sum())
print(
    "전체 수신 후 열람률:",
    round(dashboard_detail["viewed_int"].mean() * 100, 2),
    "%"
)

print("\n저장 위치:", output_dir.resolve())

[저장 완료]
상세 데이터: (66041, 32)
Offer 요약: (10, 12)
고객군 요약: (18, 8)

[주요 지표 검수]
전체 수신 건수: 66041
전체 열람 건수: 49417
전체 수신 후 열람률: 74.83 %

저장 위치: C:\Users\user\OneDrive\Desktop\새 폴더\starbucks_dashboard\dashboard_data
